# 16. Descriptors & Properties (5+ Years Interview Guide)
Deep dive into Python descriptor protocols, data vs non-data descriptors, attribute lookup precedence hierarchy, property setters/deleters, and weak references.

### Key 5-Year Interview Concepts Covered:
- **Descriptor Protocol**: `__get__`, `__set__`, `__delete__`, and `__set_name__` (PEP 487).
- **Attribute Lookup Precedence Hierarchy**: Data Descriptors > Instance `__dict__` > Non-Data Descriptors (Methods) > Class `__dict__` > `__getattr__`.
- **Managed Attributes with `@property`**: Encapsulating field validation, computed read-only properties, and property setters.
- **Weak References (`weakref`)**: Preventing memory retention cycles in descriptor storage maps.

This notebook uses the shared Fintech dataset `data/raw_transactions.csv` for interview scenario problems at the end.

In [ ]:
# Setup: Locate the Shared Dataset
import os
csv_path = 'data/raw_transactions.csv' if os.path.exists('data/raw_transactions.csv') else '../data/raw_transactions.csv'
print("Using CSV file path:", csv_path)

### 1. Managed Attributes with `@property` Getters
**Explanation**: The `@property` decorator transforms a method into a managed read-only attribute accessed via dot notation `obj.prop` without parentheses. Under the hood, `@property` creates a data descriptor on the class with a `__get__` method bound to the getter function.

**Syntax**: `@property\ndef balance(self): return self._balance`

In [ ]:
class Box:
    @property
    def size(self): return 10
print(Box().size)

### 2. Property Setters & Input Validation (`@prop.setter`)
**Explanation**: Defining `@prop.setter` attaches a setter function to the property descriptor. Whenever `obj.prop = value` is executed, the setter method runs, enabling transparent validation (e.g. rejecting negative financial amounts) without changing external public API syntax.

**Syntax**: `@balance.setter\ndef balance(self, val): if val < 0: raise ValueError; self._balance = val`

In [ ]:
class ValidatedTransaction:
    @property
    def balance(self): return self._balance
    @balance.setter
    def balance(self, value):
        if value < 0: raise ValueError('Min 0')
        self._balance = value
tx_instance = ValidatedTransaction()
tx_instance.balance = 5
print(tx_instance.balance)

### 3. Property Deleters (`@prop.deleter`)
**Explanation**: Defining `@prop.deleter` executes custom cleanup logic when `del obj.prop` is called. It can reset cached values, unbind resources, or prevent attribute deletion by raising an error.

**Syntax**: `@balance.deleter\ndef balance(self): del self._balance`

In [ ]:
class ValidatedTransaction:
    @property
    def balance(self): return self._balance
    @balance.deleter
    def balance(self): print('Deleter execution logs reached')
del ValidatedTransaction().balance

### 4. Data Descriptors Protocol
**Explanation**: A Data Descriptor is a class that implements at least `__set__()` or `__delete__()` (and usually `__get__()`). Data descriptors take precedence over instance `__dict__` during attribute lookups: even if `obj.__dict__['x']` exists, `obj.x` will invoke the data descriptor's `__get__` method.

**Syntax**: `class DataDescriptor: def __get__(self, obj, cls): ... def __set__(self, obj, val): ...`

In [ ]:
class PositiveIntegerValidator:
    def __set__(self, instance, value): instance._validated_value = value
class BoxContainer:
    validated_limit = PositiveIntegerValidator()
container = BoxContainer()
container.validated_limit = 5
print(container._validated_value)

### 5. Non-Data Descriptors Protocol
**Explanation**: A Non-Data Descriptor implements ONLY `__get__()` (no `__set__` or `__delete__`). Standard Python methods, `@classmethod`, and `@staticmethod` are implemented as non-data descriptors. If an instance defines an attribute with the same name in `obj.__dict__`, the instance attribute overrides (shadows) the non-data descriptor.

**Syntax**: `class NonDataDescriptor: def __get__(self, obj, cls): ...`

In [ ]:
class NonDataDescriptor:
    def __get__(self, instance, owner): return 'ND'
class BoxContainer:
    validated_limit = NonDataDescriptor()
print(BoxContainer().validated_limit)

### 6. Descriptor Deletion Protocol (`__delete__`)
**Explanation**: The `__delete__(self, instance)` method intercepts `del instance.attribute`. It is used to clean up external cache records, notify observers, or manage database transaction state.

**Syntax**: `def __delete__(self, instance): pass`

In [ ]:
class NonDataDescriptor:
    def __delete__(self, instance): print('Delete intercepted safely')
class BoxContainer:
    validated_limit = NonDataDescriptor()
container = BoxContainer()
del container.validated_limit

### 7. Automatic Name Binding with `__set_name__` (PEP 487)
**Explanation**: Introduced in Python 3.6, `__set_name__(self, owner, name)` is automatically called when the enclosing class is constructed. It informs the descriptor of the variable name it was assigned to (e.g. `amount = PositiveNumber()`), eliminating the need to pass the attribute name as a string manually.

**Syntax**: `def __set_name__(self, owner, name): self.name = name; self.private_name = f'_{name}'`

In [ ]:
class DescriptorClass:
    def __set_name__(self, owner, name): self.name = name
class BoxContainer:
    validated_limit = DescriptorClass()
print('Bound name:', BoxContainer.validated_limit.name)

### 8. Storing State on Instances vs Inside Descriptors
**Explanation**: Because a descriptor instance is shared across all instances of the owner class (it lives in `OwnerClass.__dict__`), NEVER store instance-specific values directly on `self` inside the descriptor. Instead, store values in `instance.__dict__[self.name]` or on `instance` using private attribute names.

**Syntax**: `def __set__(self, instance, val): instance.__dict__[self.private_name] = val`

In [ ]:
class DescriptorClass:
    def __set_name__(self, owner, name): self.name = name
    def __get__(self, instance, owner): return instance.__dict__.get(self.name)
    def __set__(self, instance, value): instance.__dict__[self.name] = value
class BoxContainer:
    validated_limit = DescriptorClass()
container = BoxContainer()
container.validated_limit = 100
print(container.validated_limit)

### 9. Reusable Validated Field Descriptors (DRY Validation)
**Explanation**: Descriptors excel at providing reusable validation logic across multiple models (e.g. `PositiveFloat`, `NonEmptyString`, `RegexField`). Instead of writing identical `@property` setters on 10 different classes, instantiate the descriptor once per field.

**Syntax**: `class Model: amount = PositiveFloat(); fee = PositiveFloat()`

In [ ]:
class DescriptorClass:
    def __set__(self, instance, value):
        if value <= 0: raise ValueError('Min 1')
        instance._v = value
class BoxContainer:
    validated_limit = DescriptorClass()
print(BoxContainer)

### 10. Modifying Class Descriptors State
**Explanation**: When accessed through the class `OwnerClass.descriptor_field`, `__get__(self, instance, owner)` is called with `instance=None`. Standard convention is to return the descriptor instance itself (`if instance is None: return self`), allowing class-level inspection.

**Syntax**: `if instance is None: return self`

In [ ]:
class BoxContainer: pass
print(BoxContainer)

### 11. Complete Attribute Lookup Precedence Hierarchy
**Explanation**: When resolving `obj.x`, Python follows strict precedence: 1. Data Descriptors on the class/MRO; 2. Instance dictionary `obj.__dict__['x']`; 3. Non-Data Descriptors (methods) on the class/MRO; 4. Class dictionary `OwnerClass.__dict__['x']`; 5. `__getattr__()` hook if defined; 6. `AttributeError`.

**Syntax**: `# Precedence: Data Descriptor -> Instance Dict -> Non-Data Descriptor -> Class Dict`

In [ ]:
class DescriptorClass: pass
class BoxContainer:
    validated_limit = DescriptorClass()
print(type(BoxContainer.__dict__['validated_limit']))

### 12. Dynamic Properties with `property()` Function
**Explanation**: Beyond decorator syntax, `@property` can be constructed programmatically using `property(fget, fset, fdel, doc)`. This is useful for dynamically attaching properties to classes in metaclasses or class factories.

**Syntax**: `cls.dynamic_field = property(getter_fn, setter_fn)`

In [ ]:
class DescriptorClass:
    def __get__(self, instance, owner): return instance, owner
class BoxContainer:
    validated_limit = DescriptorClass()
print(BoxContainer().validated_limit)

### 13. Weak References in Descriptors (`weakref.WeakKeyDictionary`)
**Explanation**: If a descriptor must store state externally rather than modifying instance `__dict__`, using a standard dictionary `self.data[instance] = val` creates a strong reference that prevents `instance` from ever being garbage collected (memory leak). Using `weakref.WeakKeyDictionary` automatically removes dictionary entries when the instance is deleted.

**Syntax**: `import weakref; self.data = weakref.WeakKeyDictionary()`

In [ ]:
import weakref
weak_map_dictionary = weakref.WeakKeyDictionary()
class DummyClass: pass
dummy_instance = DummyClass()
weak_map_dictionary[dummy_instance] = 'val'
print(weak_map_dictionary[dummy_instance])

### 14. Dynamic Property Creation via Class Decorators
**Explanation**: Class decorators can inspect class annotations and automatically attach descriptor properties at class creation time, building lightweight validation engines.

**Syntax**: `def validate_fields(cls): ...; return cls`

In [ ]:
class BoxContainer:
    def __init__(self): self._size = 5
    def get_size(self): return self._size
    size = property(get_size)
print(BoxContainer().size)

### 15. Class-Level Properties via Metaclass Descriptors
**Explanation**: To create a property on a class itself (accessed via `ClassName.prop` rather than `instance.prop`), define a standard property or descriptor on the class's metaclass.

**Syntax**: `class Meta(type): @property def version(cls): return '1.0'`

In [ ]:
class MetaclassProperty(type):
    @property
    def system_info(cls): return 'MetaInfo'
class BoxContainer(metaclass=MetaclassProperty): pass
print(BoxContainer.system_info)

## Section 3: Fintech Senior Interview Scenarios
**Explanation**: Building reusable high-performance financial validation descriptors for transaction amounts and preventing memory leaks with `WeakKeyDictionary`.


In [ ]:
# Solution:
class AmountValidator:
    def __set_name__(self, owner, name):
        self.name = f'_{name}'
    def __get__(self, instance, owner):
        return getattr(instance, self.name, 0.0)
    def __set__(self, instance, value):
        if value < 0.0:
            raise ValueError('Amount cannot be negative')
        setattr(instance, self.name, value)

class Account:
    balance = AmountValidator()
    def __init__(self, val): self.balance = val
    
acc = Account(150.0)
print('Validated balance:', acc.balance)


### Q2: Memory Leak Prevention with `weakref` Descriptors
**Explanation**: **Scenario**: Implement a descriptor using `weakref.WeakKeyDictionary` to track external transaction audit logs and verify that deleting transaction instances frees memory immediately.

**Syntax**: `self.storage = weakref.WeakKeyDictionary()`

In [ ]:
# Solution:
import weakref
class WeakStore:
    def __init__(self):
        self.data = weakref.WeakKeyDictionary()
        
ws = WeakStore()
print('WeakKeyDictionary instantiated safely:', ws.data)
